In [25]:
# %pip install yfinance
# %pip install edgartools

In [26]:
import os

files_to_delete = [
    '../datasets/company_dim.csv',
    '../datasets/prices_fact.csv',
    '../datasets/wall_street_estimate_fact.csv',
    '../datasets/quarterly/quarterly_income_statements_fact.csv',
    '../datasets/quarterly/quarterly_cash_flows_fact.csv',
    '../datasets/quarterly/quarterly_balance_sheets_fact.csv',
    '../datasets/yearly/yearly_income_statements_fact.csv',
    '../datasets/yearly/yearly_cash_flows_fact.csv',
    '../datasets/yearly/yearly_balance_sheets_fact.csv'
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted: {file}")
    else:
        print(f"File not found, skipping: {file}")

File not found, skipping: ../datasets/company_dim.csv
File not found, skipping: ../datasets/prices_fact.csv
File not found, skipping: ../datasets/wall_street_estimate_fact.csv
File not found, skipping: ../datasets/quarterly/quarterly_income_statements_fact.csv
File not found, skipping: ../datasets/quarterly/quarterly_cash_flows_fact.csv
File not found, skipping: ../datasets/quarterly/quarterly_balance_sheets_fact.csv
File not found, skipping: ../datasets/yearly/yearly_income_statements_fact.csv
File not found, skipping: ../datasets/yearly/yearly_cash_flows_fact.csv
File not found, skipping: ../datasets/yearly/yearly_balance_sheets_fact.csv


# Dashboard

## Valuation history
- Show the chart of 5 year average
- the earliest value shown on the chart is also averaged by the past 5 years


---


## Financials

1.   Revenue
2.   Operating income
3.   Net income
4.   EPS
5.   Operating cash flow
6.   Free cash flow


---


## Revenue & Expenses Breakdown

*   Revenue +
*   Gross profit +
*   Other income +
*   Cost of revenue -
*   Net income +
*   Expenses -
*   SG&A -
*   R&D -


---


## Balance sheet
### Assets
Current Assets

*   Cash & Short-Term Investments
*   Receivables
*   Other Current Assets

Non-Current Assets


*   PP&E
*   Intangibles
*   Other Non-Current Assets

### Liabilities

Current Liabilities


*   Accounts Payable
*   Accrued Liabilities
*   Other Current Liabilities


Non-Current Liabilities


*   Long-Term Debt
*   Other Non-Current Liabilities


---


# Free Cash Flow Analysis

## Gross Margin

Gross Profit / Total Revenue

## Operating Margin

Operating Income / Total Revenue

## Net Margin

Net Income / Total Revenue

## FCF Margin

Free Cash Flow / Total Revenue

## ROE (Return on Equity)

Net Income / equity

## ROA (Return on Assets)

Net Income / Average Total Assets

## ROIC (Return on Invested Capital)

NOPAT(ebit * (1 - tax rate(tax provision / pretax income))) / Invested Capital (total debt + stockholders equity - cash and equiv)

## ROCE (Return on Capital Employed)

EBIT / Capital Employed(total assets - current liabilities)

# PEG Ratio
**Price/earnings to growth ratio** of 1 means the stock is reasonably priced, meaning the company is fairly priced compared to it's growth rate:


*   **PEG Over 1:** Overvalued
*   **PEG Around 1:** Fairly valued
*   **PEG Below 1:** Undervalued

**Note:** the PEG ratio most accurate for mature companies with positive earnings, stable, and measurable growth trajectories. But it's often unreliable for cyclical stocks

# Formula
PEG Ratio = (P/E Ratio) / (Expected Earnings Growth Rate)

Where:

*   **P/E Ratio** = Market Price per Share / Earnings per Share (EPS)
*   **Expected Earnings Growth Rate** = the anticipated annual growth rate of the company's EPS, usually projected for the next 1-3 years. (forward price per earnings / growth)


---


# P/B Ratio
**Price-to-Book ratio** compares a company's market value to its book value (net assets). A ratio of 1 means the stock is trading at its book value, indicating it may be fairly valued relative to its net asset worth:

*   **P/B Over 1:** Overvalued (market price exceeds book value)
*   **P/B Around 1:** Fairly valued
*   **P/B Below 1:** Undervalued (market price is less than book value)

**Note:** The P/B ratio is most accurate for asset-heavy companies like banks, real estate, and manufacturing firms.  It is often unreliable for companies with significant intangible assets (like tech firms) or those in financial distress.

# Formula
P/B Ratio = Market Price per Share / Book Value per Share

Where:

**Book Value per Share** = (Total Shareholders' Equity - Preferred Equity) / Total Number of Outstanding Common Shares

In [27]:
import pandas as pd
import numpy as np
import yfinance as yf
import edgar as ed

# Use your name and email (required by SEC)
ed.set_identity("John Doe john.doe@company.com")

In [28]:
my_tickers = ['GOOGL', 'META', 'AMZN']

In [29]:
company_data = []

for ticker in my_tickers:
  tickers_data = yf.Ticker(ticker)
  info = tickers_data.info

  company_data.append({
          'company_name': info.get('shortName'),
          'ticker': info.get('symbol'),
          'type': info.get('quoteType'),
          'country': info.get('country'),
          'region': info.get('region'),
          'exchange': info.get('fullExchangeName'),
          'currency': info.get('currency'),
          'industry': info.get('industry'),
          'sector': info.get('sector'),
          'outstanding_shares': info.get('sharesOutstanding')
      })

df = pd.DataFrame(company_data)
df.to_csv('../datasets/company_dim.csv', index=False)

In [30]:
price_history = yf.download(my_tickers, period='10y', auto_adjust=True) # Auto_adjust adjusts the historical price if the stock did a stock split or pays out dividends

# Convert from horizontal data to vertical data
# Reset index is used to avoid treating Date and Ticker column as the index
price_history_stacked = price_history.stack(future_stack=True).reset_index()

cols_to_round = ['Close', 'High', 'Low', 'Open']
price_history_stacked[cols_to_round] = price_history_stacked[cols_to_round].round(3)

price_history_stacked.to_csv('../datasets/prices_fact.csv', index=False)

[*********************100%***********************]  3 of 3 completed


In [31]:
wall_street_estimate_fact = pd.DataFrame()

for ticker in my_tickers:
    tickers_data = yf.Ticker(ticker)
    targets = tickers_data.analyst_price_targets

    wall_street_estimate_df = pd.DataFrame([targets])
    wall_street_estimate_df['ticker'] = ticker

    wall_street_estimate_fact = pd.concat([wall_street_estimate_fact, wall_street_estimate_df])

wall_street_estimate_fact.to_csv('../datasets/wall_street_estimate_fact.csv', index=False)

date mapping functions

In [32]:
# used to map quarters with the 10-Q and 10-K SEC filings date
def build_quarter_date_map(company):
    tenqs = company.get_filings(form='10-Q')
    tenks = company.get_filings(form='10-K')

    q_data = []
    for filing in tenqs:
        current_date = pd.to_datetime(filing.filing_date)
        if not q_data:
            q_data.append((filing.filing_date, '10-Q'))
        else:
            last_valid_date = pd.to_datetime(q_data[-1][0])
            if (last_valid_date - current_date).days >= 80:
                q_data.append((filing.filing_date, '10-Q'))
        if len(q_data) == 30:
            break

    k_data = []
    for filing in tenks:
        current_date = pd.to_datetime(filing.filing_date)
        if not k_data:
            k_data.append((filing.filing_date, '10-K'))
        else:
            last_valid_date = pd.to_datetime(k_data[-1][0])
            if (last_valid_date - current_date).days >= 300:
                k_data.append((filing.filing_date, '10-K'))
        if len(k_data) == 10:
            break

    combined = sorted(q_data + k_data, key=lambda x: x[0], reverse=True)
    return combined

In [33]:
# used to map quarters with the 10-K SEC filings date
def build_annual_date_map(company):
    combined = build_quarter_date_map(company)

    annual_dates = []
    for date, form_type in combined:
        if form_type != '10-K':
            continue

        filing_date = pd.to_datetime(date)

        if filing_date.month < 7:
            q3_date = None
            for q_date, q_form in combined:
                q_date_dt = pd.to_datetime(q_date)
                days_before = (filing_date - q_date_dt).days
                if q_form == '10-Q' and 60 <= days_before <= 270:
                    q3_date = q_date_dt
                    break
            annual_dates.append((q3_date or filing_date).strftime('%Y-%m-%d'))
        else:
            annual_dates.append(filing_date.strftime('%Y-%m-%d'))

    return annual_dates

# Quarterly

In [34]:
quarterly_income_statements_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    income_statements_data_df = tickers_data.income_statement(periods=40, annual=False).to_dataframe()
    income_statements_data_df = income_statements_data_df.iloc[:, np.r_[0, 6:len(income_statements_data_df.columns)]].fillna(0).round(3)

    combined = build_quarter_date_map(company)
    quarter_cols = income_statements_data_df.columns[1:]
    date_map = {
        quarter_cols[i]: pd.to_datetime(combined[i][0]).strftime('%Y-%m-%d')
        for i in range(min(len(quarter_cols), len(combined)))
    }
    income_statements_data_df = income_statements_data_df.rename(columns=date_map)

    long_df = income_statements_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)
    quarterly_income_statements_fact = pd.concat([quarterly_income_statements_fact, long_df])

long_isf = quarterly_income_statements_fact.rename(columns={'label': 'name'})
sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/quarterly/quarterly_income_statements_fact.csv', index=False)

Data quality issue: Q1 (-1.44B) > YTD_6M (-2.92B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q2 derivation
Data quality issue: Q1 (5.21B) > YTD_6M (2.79B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q2 derivation
Data quality issue: YTD_6M (-2.92B) > YTD_9M (-7.14B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_6M (2.79B) > YTD_9M (-2.34B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_6M (3.43B) > YTD_9M (2.73B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q4 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q4 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEqu

In [35]:
quarterly_cash_flows_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    cash_flows_data_df = tickers_data.cashflow_statement(periods=40, annual=False).to_dataframe()
    cash_flows_data_df = cash_flows_data_df.iloc[:, np.r_[0, 6:len(cash_flows_data_df.columns)]].fillna(0).round(3)

    combined = build_quarter_date_map(company)
    quarter_cols = cash_flows_data_df.columns[1:]
    date_map = {
        quarter_cols[i]: pd.to_datetime(combined[i][0]).strftime('%Y-%m-%d')
        for i in range(min(len(quarter_cols), len(combined)))
    }
    cash_flows_data_df = cash_flows_data_df.rename(columns=date_map)

    long_df = cash_flows_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)
    quarterly_cash_flows_fact = pd.concat([quarterly_cash_flows_fact, long_df])

long_isf = quarterly_cash_flows_fact.rename(columns={'label': 'name'})
sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/quarterly/quarterly_cash_flows_fact.csv', index=False)

Data quality issue: Q1 (-1.44B) > YTD_6M (-2.92B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q2 derivation
Data quality issue: Q1 (5.21B) > YTD_6M (2.79B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q2 derivation
Data quality issue: YTD_6M (-2.92B) > YTD_9M (-7.14B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_6M (2.79B) > YTD_9M (-2.34B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_6M (3.43B) > YTD_9M (2.73B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q3 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q4 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease, skipping Q4 derivation
Data quality issue: YTD_9M (-0.28B) > FY (-1.80B) for us-gaap:CashAndCashEqu

In [36]:
quarterly_balance_sheets_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    balance_sheets_data_df = tickers_data.balance_sheet(periods=40, annual=False).to_dataframe()
    balance_sheets_data_df = balance_sheets_data_df.iloc[:, np.r_[0, 6:len(balance_sheets_data_df.columns)]].fillna(0).round(3)

    combined = build_quarter_date_map(company)
    quarter_cols = balance_sheets_data_df.columns[1:]
    date_map = {
        quarter_cols[i]: pd.to_datetime(combined[i][0]).strftime('%Y-%m-%d')
        for i in range(min(len(quarter_cols), len(combined)))
    }
    balance_sheets_data_df = balance_sheets_data_df.rename(columns=date_map)

    long_df = balance_sheets_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)
    quarterly_balance_sheets_fact = pd.concat([quarterly_balance_sheets_fact, long_df])

long_isf = quarterly_balance_sheets_fact.rename(columns={'label': 'name'})
sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/quarterly/quarterly_balance_sheets_fact.csv', index=False)

# Yearly

In [37]:
yearly_income_statements_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    income_statements_data_df = tickers_data.income_statement(periods=10, annual=True).to_dataframe()

    income_statements_data_df = income_statements_data_df.iloc[:, np.r_[0, 6:len(income_statements_data_df.columns)]].fillna(0).round(3)

    annual_dates = build_annual_date_map(company)
    year_cols = income_statements_data_df.columns[1:]
    date_map = {
        year_cols[i]: annual_dates[i]
        for i in range(min(len(year_cols), len(annual_dates)))
    }
    income_statements_data_df = income_statements_data_df.rename(columns=date_map)

    long_df = income_statements_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)

    yearly_income_statements_fact = pd.concat([yearly_income_statements_fact, long_df])

long_isf = yearly_income_statements_fact.rename(columns={'label': 'name'})

sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/yearly/yearly_income_statements_fact.csv', index=False)

In [38]:
yearly_cash_flows_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    cash_flows_data_df = tickers_data.cashflow_statement(periods=10, annual=True).to_dataframe()

    cash_flows_data_df = cash_flows_data_df.iloc[:, np.r_[0, 6:len(cash_flows_data_df.columns)]].fillna(0).round(3)

    annual_dates = build_annual_date_map(company)
    year_cols = cash_flows_data_df.columns[1:]
    date_map = {
        year_cols[i]: annual_dates[i]
        for i in range(min(len(year_cols), len(annual_dates)))
    }
    cash_flows_data_df = cash_flows_data_df.rename(columns=date_map)

    long_df = cash_flows_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)

    yearly_cash_flows_fact = pd.concat([yearly_cash_flows_fact, long_df])

long_isf = yearly_cash_flows_fact.rename(columns={'label': 'name'})

sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/yearly/yearly_cash_flows_fact.csv', index=False)

In [39]:
yearly_balance_sheets_fact = pd.DataFrame()

for ticker in my_tickers:
    company = ed.Company(ticker)
    tickers_data = company.get_facts()
    balance_sheets_data_df = tickers_data.balance_sheet(periods=10, annual=True).to_dataframe()

    balance_sheets_data_df = balance_sheets_data_df.iloc[:, np.r_[0, 6:len(balance_sheets_data_df.columns)]].fillna(0).round(3)

    annual_dates = build_annual_date_map(company)
    year_cols = balance_sheets_data_df.columns[1:]
    date_map = {
        year_cols[i]: annual_dates[i]
        for i in range(min(len(year_cols), len(annual_dates)))
    }
    balance_sheets_data_df = balance_sheets_data_df.rename(columns=date_map)

    long_df = balance_sheets_data_df.melt(id_vars=['label'], var_name='date', value_name='value')
    long_df.insert(0, 'ticker', ticker)

    yearly_balance_sheets_fact = pd.concat([yearly_balance_sheets_fact, long_df])

long_isf = yearly_balance_sheets_fact.rename(columns={'label': 'name'})

sorted_isf = long_isf.sort_values(by=['ticker', 'name', 'date'], ascending=[True, False, False]).reset_index(drop=True)
sorted_isf = sorted_isf.drop_duplicates(subset=['ticker', 'name', 'date'], keep='last')
sorted_isf.to_csv('../datasets/yearly/yearly_balance_sheets_fact.csv', index=False)

# quarterly (deprecated)

In [40]:
# ttm_income_statements_fact = pd.DataFrame()

# for ticker in my_tickers:
#   tickers_data = yf.Ticker(ticker)
#   income_statements_data_df = pd.DataFrame(tickers_data.quarterly_income_stmt)

#   income_statements_data_df = income_statements_data_df.iloc[:, :4].astype(float).round(3)

#   income_statements_data_df_with_ticker = pd.concat({ticker: income_statements_data_df})

#   ttm_income_statements_fact = pd.concat([ttm_income_statements_fact, income_statements_data_df_with_ticker])

# finished_company_income = ttm_income_statements_fact.stack().reset_index()
# finished_company_income.columns = ['ticker', 'name', 'date', 'value']
# finished_company_income.to_csv('../datasets/ttm/ttm_income_statements_fact.csv', index=False)

In [41]:
# ttm_cash_flow_fact = pd.DataFrame()

# for ticker in my_tickers:
#   tickers_data = yf.Ticker(ticker)
#   cash_flow_data_df = pd.DataFrame(tickers_data.quarterly_cash_flow)

#   cash_flow_data_df = cash_flow_data_df.iloc[:, :4].astype(float).round(3)

#   cash_flow_data_df_with_ticker = pd.concat({ticker: cash_flow_data_df})

#   ttm_cash_flow_fact = pd.concat([ttm_cash_flow_fact, cash_flow_data_df_with_ticker])

# finished_company_cash_flow = ttm_cash_flow_fact.stack().reset_index()
# finished_company_cash_flow.columns = ['ticker', 'name', 'date', 'value']
# finished_company_cash_flow.to_csv('../datasets/ttm/ttm_cash_flow_fact.csv', index=False)

In [42]:
# ttm_balance_sheet_fact = pd.DataFrame()

# for ticker in my_tickers:
#     tickers_data = yf.Ticker(ticker)
#     balance_sheet_df = pd.DataFrame(tickers_data.quarterly_balance_sheet)

#     balance_sheet_df = balance_sheet_df.iloc[:, :4].astype(float).round(3)

#     balance_sheet_with_ticker = pd.concat({ticker: balance_sheet_df})

#     ttm_balance_sheet_fact = pd.concat([ttm_balance_sheet_fact, balance_sheet_with_ticker])

# finished_balance_sheet = ttm_balance_sheet_fact.stack().reset_index()
# finished_balance_sheet.columns = ['ticker', 'name', 'date', 'value']
# finished_balance_sheet.to_csv('../datasets/ttm/ttm_balance_sheet_fact.csv', index=False)

# Yearly (deprecated)

In [43]:
# yearly_income_statements_fact = pd.DataFrame()

# for ticker in my_tickers:
#   tickers_data = yf.Ticker(ticker)
#   income_statements_data_df = pd.DataFrame(tickers_data.income_stmt)

#   income_statements_data_df = income_statements_data_df.iloc[:, :4].astype(float).round(3)

#   income_statements_data_df_with_ticker = pd.concat({ticker: income_statements_data_df})

#   yearly_income_statements_fact = pd.concat([yearly_income_statements_fact, income_statements_data_df_with_ticker])

# finished_company_income = yearly_income_statements_fact.stack().reset_index()
# finished_company_income.columns = ['ticker', 'name', 'date', 'value']
# finished_company_income.to_csv('../datasets/yearly/yearly_income_statements_fact.csv', index=False)

In [44]:
# yearly_cash_flow_fact = pd.DataFrame()

# for ticker in my_tickers:
#   tickers_data = yf.Ticker(ticker)
#   cash_flow_data_df = pd.DataFrame(tickers_data.cash_flow)

#   cash_flow_data_df = cash_flow_data_df.iloc[:, :4].astype(float).round(3)

#   cash_flow_data_df_with_ticker = pd.concat({ticker: cash_flow_data_df})

#   yearly_cash_flow_fact = pd.concat([yearly_cash_flow_fact, cash_flow_data_df_with_ticker])

# finished_company_cash_flow = yearly_cash_flow_fact.stack().reset_index()
# finished_company_cash_flow.columns = ['ticker', 'name', 'date', 'value']
# finished_company_cash_flow.to_csv('../datasets/yearly/yearly_cash_flow_fact.csv', index=False)

In [45]:
# yearly_balance_sheet_fact = pd.DataFrame()

# for ticker in my_tickers:
#     tickers_data = yf.Ticker(ticker)
#     balance_sheet_df = pd.DataFrame(tickers_data.balance_sheet)

#     balance_sheet_df = balance_sheet_df.iloc[:, :4].astype(float).round(3)

#     balance_sheet_with_ticker = pd.concat({ticker: balance_sheet_df})

#     yearly_balance_sheet_fact = pd.concat([yearly_balance_sheet_fact, balance_sheet_with_ticker])

# finished_balance_sheet = yearly_balance_sheet_fact.stack().reset_index()
# finished_balance_sheet.columns = ['ticker', 'name', 'date', 'value']
# finished_balance_sheet.to_csv('../datasets/yearly/yearly_balance_sheet_fact.csv', index=False)